In [1]:
from dotenv import load_dotenv

import os

load_dotenv()

project_name = "wanted_2nd_agent_study"
os.environ["LANGSMITH_PROJECT"] = project_name

In [2]:
from langchain.agents import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Dict, Optional, Any

In [3]:
# 1. 우리가 만든 툴킷 가져오기
from dolmary_schedule_toolkit import DolmaryScheduleToolkit
from dolmary_schedule_toolkit.toolkit import AddToDoTool

In [4]:
toolkits = DolmaryScheduleToolkit()
tools = toolkits.get_tools()
tools

[AddToDoTool(), ViewToDoTool(), DeleteToDoTool(), SuccessedToDoTool()]

In [5]:
# 2. 모델 설정
model = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature=0
)

# 3. 프롬프트 설정
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 돌마리 스케줄을 관리하는 AI 어시스턴트야. 주어진 도구를 잘 활용해서 사용자에게 응답을 해줘. 시간을 등록할 때는 HH:MM 형식으로 등록해줘."),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
        ("placeholder", "{agent_scratchpad}")
    ]
)

# 4. 에이전트 생성
agent = create_openai_tools_agent(
    model,
    tools,
    prompt
)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from typing import Dict
from langchain_core.chat_history import InMemoryChatMessageHistory

In [7]:
stores : Dict[str, InMemoryChatMessageHistory] = {}

def get_store(session_id: str) -> RunnableWithMessageHistory:
    if session_id not in stores:
        stores[session_id] = InMemoryChatMessageHistory()
    return stores[session_id]

agent_history = RunnableWithMessageHistory(
    executor,
    lambda session_id: get_store(session_id),
    input_messages_key="question",
    history_messages_key="chat_history"
)

In [8]:
config = {"configurable": {"session_id": "user-123"}}
agent_history.invoke({"question": "오늘 돌마리 운동회 12시 30분에 파워 댄스 일정 등록해줘."}, config=config)



> Entering new AgentExecutor chain...

Invoking: `add_todo` with `{'time': '12:30', 'schedule': '파워 댄스'}`


12:30에 파워 댄스이 돌마리 스케줄에 등록되었습니다.오늘 12시 30분에 파워 댄스 일정이 돌마리 스케줄에 등록되었습니다.

> Finished chain.


{'question': '오늘 돌마리 운동회 12시 30분에 파워 댄스 일정 등록해줘.',
 'chat_history': [],
 'output': '오늘 12시 30분에 파워 댄스 일정이 돌마리 스케줄에 등록되었습니다.'}

In [9]:
agent_history.invoke({"question": "오늘 돌마리 스케줄 전체 보여줘."}, config=config)



> Entering new AgentExecutor chain...

Invoking: `view_todos` with `{}`


등록된 스케줄 목록: 
시간: 12:30, 일정: 파워 댄스
오늘 돌마리 스케줄 전체는 다음과 같습니다.
- 12:30 파워 댄스

> Finished chain.


{'question': '오늘 돌마리 스케줄 전체 보여줘.',
 'chat_history': [HumanMessage(content='오늘 돌마리 운동회 12시 30분에 파워 댄스 일정 등록해줘.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='오늘 12시 30분에 파워 댄스 일정이 돌마리 스케줄에 등록되었습니다.', additional_kwargs={}, response_metadata={})],
 'output': '오늘 돌마리 스케줄 전체는 다음과 같습니다.\n- 12:30 파워 댄스'}

In [11]:
agent_history.invoke({"question": "오늘 12시 30분 파워 댄스 일정을 삭제해줘."}, config=config)



> Entering new AgentExecutor chain...

Invoking: `delete_todo` with `{'time': '12:30', 'schedule': '파워 댄스'}`


12:30에 파워 댄스 일정이 삭제되었습니다.오늘 12시 30분에 등록된 파워 댄스 일정이 삭제되었습니다. 다른 일정 관리도 도와드릴까요?

> Finished chain.


{'question': '오늘 12시 30분 파워 댄스 일정을 삭제해줘.',
 'chat_history': [HumanMessage(content='오늘 돌마리 운동회 12시 30분에 파워 댄스 일정 등록해줘.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='오늘 12시 30분에 파워 댄스 일정이 돌마리 스케줄에 등록되었습니다. 더 도와드릴 일이 있을까요?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='오늘 돌마리 스케줄 전체 보여줘.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='오늘 돌마리 스케줄 전체는 다음과 같습니다.\n- 12:30 파워 댄스\n다른 일정도 확인하거나 추가할까요?', additional_kwargs={}, response_metadata={})],
 'output': '오늘 12시 30분에 등록된 파워 댄스 일정이 삭제되었습니다. 다른 일정 관리도 도와드릴까요?'}

In [10]:
agent_history.invoke({"question": "오늘 12시 30분 파워 댄스 일정 완료으로 체크 해줘"}, config=config)



> Entering new AgentExecutor chain...

Invoking: `successed_todo` with `{'time': '12:30', 'schedule': '파워 댄스'}`


12:30에 파워 댄스 일정이 성공적으로 완료되었습니다.
12:30에 파워 댄스 일정이 성공적으로 완료되었습니다.오늘 12시 30분 파워 댄스 일정이 완료로 체크되었습니다.

> Finished chain.


{'question': '오늘 12시 30분 파워 댄스 일정 완료으로 체크 해줘',
 'chat_history': [HumanMessage(content='오늘 돌마리 운동회 12시 30분에 파워 댄스 일정 등록해줘.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='오늘 12시 30분에 파워 댄스 일정이 돌마리 스케줄에 등록되었습니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='오늘 돌마리 스케줄 전체 보여줘.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='오늘 돌마리 스케줄 전체는 다음과 같습니다.\n- 12:30 파워 댄스', additional_kwargs={}, response_metadata={})],
 'output': '오늘 12시 30분 파워 댄스 일정이 완료로 체크되었습니다.'}